In [2]:
import polars as pl


In [14]:
COLUMNS_TO_KEEP = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "trip_distance",
    "passenger_count",
    "fare_amount",
    "total_amount",
]


In [34]:


df = pl.read_parquet(
    "../data/raw/yellow_tripdata_2025-01.parquet",columns=COLUMNS_TO_KEEP
)
df.describe()

statistic,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,trip_distance,passenger_count,fare_amount,total_amount
str,str,str,f64,f64,f64,f64,f64
"""count""","""3475226""","""3475226""",3.475226e6,3.475226e6,2.935077e6,3.475226e6,3.475226e6
"""null_count""","""0""","""0""",0.0,0.0,540149.0,0.0,0.0
"""mean""","""2025-01-17 11:02:55.910964""","""2025-01-17 11:17:56.997901""",165.191576,5.855126,1.297859,17.081803,25.611292
"""std""",null,null,64.529483,564.6016,0.75075,463.472918,463.658478
"""min""","""2024-12-31 20:47:55""","""2024-12-18 07:52:40""",1.0,0.0,0.0,-900.0,-901.0
"""25%""","""2025-01-10 07:59:01""","""2025-01-10 08:15:29""",132.0,0.98,1.0,8.6,15.2
"""50%""","""2025-01-17 15:41:34""","""2025-01-17 15:59:34""",162.0,1.67,1.0,12.11,19.95
"""75%""","""2025-01-24 19:34:06""","""2025-01-24 19:48:31""",234.0,3.1,1.0,19.5,27.78
"""max""","""2025-02-01 00:00:44""","""2025-02-01 23:44:11""",265.0,276423.57,9.0,863372.12,863380.37


In [35]:
df.shape

(3475226, 7)

In [36]:
df=df.rename({"tpep_pickup_datetime":"pickup_dt","tpep_dropoff_datetime":"dropoff_dt"})

In [37]:
df.head()

pickup_dt,dropoff_dt,PULocationID,trip_distance,passenger_count,fare_amount,total_amount
datetime[μs],datetime[μs],i32,f64,i64,f64,f64
2025-01-01 00:18:38,2025-01-01 00:26:59,229,1.6,1,10.0,18.0
2025-01-01 00:32:40,2025-01-01 00:35:13,236,0.5,1,5.1,12.12
2025-01-01 00:44:04,2025-01-01 00:46:01,141,0.6,1,5.1,12.1
2025-01-01 00:14:27,2025-01-01 00:20:01,244,0.52,3,7.2,9.7
2025-01-01 00:21:34,2025-01-01 00:25:06,244,0.66,3,5.8,8.3


In [38]:
df.describe()

statistic,pickup_dt,dropoff_dt,PULocationID,trip_distance,passenger_count,fare_amount,total_amount
str,str,str,f64,f64,f64,f64,f64
"""count""","""3475226""","""3475226""",3.475226e6,3.475226e6,2.935077e6,3.475226e6,3.475226e6
"""null_count""","""0""","""0""",0.0,0.0,540149.0,0.0,0.0
"""mean""","""2025-01-17 11:02:55.910964""","""2025-01-17 11:17:56.997901""",165.191576,5.855126,1.297859,17.081803,25.611292
"""std""",null,null,64.529483,564.6016,0.75075,463.472918,463.658478
"""min""","""2024-12-31 20:47:55""","""2024-12-18 07:52:40""",1.0,0.0,0.0,-900.0,-901.0
"""25%""","""2025-01-10 07:59:01""","""2025-01-10 08:15:29""",132.0,0.98,1.0,8.6,15.2
"""50%""","""2025-01-17 15:41:34""","""2025-01-17 15:59:34""",162.0,1.67,1.0,12.11,19.95
"""75%""","""2025-01-24 19:34:06""","""2025-01-24 19:48:31""",234.0,3.1,1.0,19.5,27.78
"""max""","""2025-02-01 00:00:44""","""2025-02-01 23:44:11""",265.0,276423.57,9.0,863372.12,863380.37


In [39]:
df=df.drop_nulls(subset=['pickup_dt'])

In [40]:
df.shape

(3475226, 7)

In [41]:
df=df.filter(
    (pl.col("trip_distance")>0) &
    (pl.col("fare_amount")>0 ) &
    (pl.col("PULocationID").is_between(1,263)) &
    (pl.col("pickup_dt").dt.year()==2025)
)

In [42]:
df.shape

(3245616, 7)

In [43]:
df=df.with_columns(
    pl.col("pickup_dt")
    .dt.truncate("1h")
    .alias("pickup_hour")
)

In [44]:
agg_df=df.group_by(["pickup_hour","PULocationID"]).len().rename({"len":"trip_count"})

In [45]:
agg_df[13500:135510]

pickup_hour,PULocationID,trip_count
datetime[μs],i32,u32
2025-01-14 22:00:00,256,2
2025-01-29 17:00:00,10,1
2025-01-19 13:00:00,186,277
2025-01-29 12:00:00,54,1
2025-01-05 06:00:00,170,8
…,…,…
2025-01-31 07:00:00,232,10
2025-01-06 14:00:00,53,1
2025-01-28 05:00:00,20,2


In [63]:
agg_df.sort(['pickup_hour']).tail()

pickup_hour,PULocationID,trip_count
datetime[μs],i32,u32
2025-01-31 23:00:00,65,15
2025-01-31 23:00:00,45,43
2025-01-31 23:00:00,186,113
2025-01-31 23:00:00,215,1
2025-02-01 00:00:00,113,1


In [46]:
agg_df.shape

(91095, 3)

In [47]:
all_hours=pl.datetime_range(
    start=df['pickup_dt'].min().replace(minute=0,second=0,microsecond=0),
    end=df['pickup_dt'].max().replace(minute=0,second=0,microsecond=0),
    interval='1h',
    eager=True
).alias("pickup_hour")


In [48]:
all_hours

pickup_hour
datetime[μs]
2025-01-01 00:00:00
2025-01-01 01:00:00
2025-01-01 02:00:00
2025-01-01 03:00:00
2025-01-01 04:00:00
…
2025-01-31 20:00:00
2025-01-31 21:00:00
2025-01-31 22:00:00


In [49]:
all_zones=pl.Series("PULocationID",list(range(1,264)))


In [50]:
all_zones

PULocationID
i64
1
2
3
4
5
…
259
260
261


In [52]:
type(all_hours)

polars.series.series.Series

In [57]:
all_hours.shape

(745,)

In [53]:
type(all_hours.to_frame())

polars.dataframe.frame.DataFrame

In [55]:
full_grid=all_hours.to_frame().join(all_zones.to_frame(),how='cross')

In [56]:
full_grid.shape

(195935, 2)

In [59]:
result=(
    full_grid
    .join(agg_df,on=['pickup_hour',"PULocationID"],how='left')
    .with_columns(pl.col("trip_count").fill_null(0))
    .sort(['pickup_hour',"PULocationID"])
)

In [76]:
def add_time_features(df:pl.DataFrame)->pl.DataFrame:
    time_df=df.with_columns(
        [
            pl.col("pickup_hour")
            .dt.hour()
            .alias("hour_of_day"),

           pl.col("pickup_hour")
            .dt.weekday()
            .alias("day_of_week"),

            pl.col("pickup_hour")
            .dt.month()
            .alias("month"),

            (pl.col("pickup_hour").dt.weekday()>=5)
            .alias("is_weekend"),

            pl.col("pickup_hour")
            .dt.hour()
            .is_in([7,8,9,17,18,19])
            .alias("is_rush_hour"),
        ]
    )
    return time_df

In [77]:
def add_lag_features(df:pl.DataFrame)->pl.DataFrame:
    lag_df=df.with_columns(
        [
            pl.col("trip_count")
            .shift(1)
            .over("PULocationID")
            .alias("lag_1h"),

            pl.col("trip_count")
            .shift(24)
            .over("PULocationID")
            .alias("lag_24h"),

            pl.col("trip_count")
            .shift(168)
            .over("PULocationID")
            .alias("lag_168h"),
        ]
    )
    return lag_df

In [78]:
def add_rolling_features(df:pl.DataFrame)->pl.DataFrame:
    rolling_df=df.with_columns(
        [
            pl.col("trip_count")
            .shift(1)
            .rolling_mean(window_size=3)
            .over("PULocationID")
            .alias("rolling_mean_3h"),

            pl.col("trip_count")
            .shift(1)
            .rolling_mean(window_size=24)
            .over("PULocationID")
            .alias("rolling_mean_24h")
        ]
    )
    return rolling_df

In [79]:

def drop_nulls(df:pl.DataFrame)->pl.DataFrame:
    df=df.drop_nulls(subset=[
        'lag_1h',
        'lag_24h',
        'lag_168h',
        'rolling_mean_3h',
        'rolling_mean_24h'
        ])
    return df

In [80]:
TARGET_COL = "trip_count"
LOCATION_COL = "PULocationID"
TIME_COL = "pickup_hour"

In [81]:
def build_features(df:pl.DataFrame)->pl.DataFrame:
    df=df.sort([LOCATION_COL,TIME_COL])
    df=add_time_features(df)
    df=add_lag_features(df)
    df=add_rolling_features(df)
    df=drop_nulls(df)
    return df

In [82]:
featured_df=build_features(result)

In [83]:
featured_df

pickup_hour,PULocationID,trip_count,hour_of_day,day_of_week,month,is_weekend,is_rush_hour,lag_1h,lag_24h,lag_168h,rolling_mean_3h,rolling_mean_24h
datetime[μs],i64,u32,i8,i8,i8,bool,bool,u32,u32,u32,f64,f64
2025-01-08 00:00:00,1,0,0,3,1,false,false,0,0,0,0.0,0.083333
2025-01-08 01:00:00,1,0,1,3,1,false,false,0,0,0,0.0,0.083333
2025-01-08 02:00:00,1,0,2,3,1,false,false,0,0,0,0.0,0.083333
2025-01-08 03:00:00,1,0,3,3,1,false,false,0,0,0,0.0,0.083333
2025-01-08 04:00:00,1,0,4,3,1,false,false,0,0,0,0.0,0.083333
…,…,…,…,…,…,…,…,…,…,…,…,…
2025-01-31 20:00:00,263,131,20,5,1,true,false,182,123,151,196.333333,99.541667
2025-01-31 21:00:00,263,128,21,5,1,true,false,131,106,133,180.0,99.875
2025-01-31 22:00:00,263,162,22,5,1,true,false,128,81,106,147.0,100.791667


In [85]:
featured_df.filter(
    pl.col("PULocationID") == 1
)

pickup_hour,PULocationID,trip_count,hour_of_day,day_of_week,month,is_weekend,is_rush_hour,lag_1h,lag_24h,lag_168h,rolling_mean_3h,rolling_mean_24h
datetime[μs],i64,u32,i8,i8,i8,bool,bool,u32,u32,u32,f64,f64
2025-01-08 00:00:00,1,0,0,3,1,false,false,0,0,0,0.0,0.083333
2025-01-08 01:00:00,1,0,1,3,1,false,false,0,0,0,0.0,0.083333
2025-01-08 02:00:00,1,0,2,3,1,false,false,0,0,0,0.0,0.083333
2025-01-08 03:00:00,1,0,3,3,1,false,false,0,0,0,0.0,0.083333
2025-01-08 04:00:00,1,0,4,3,1,false,false,0,0,0,0.0,0.083333
…,…,…,…,…,…,…,…,…,…,…,…,…
2025-01-31 20:00:00,1,0,20,5,1,true,false,0,0,0,0.0,0.0
2025-01-31 21:00:00,1,0,21,5,1,true,false,0,0,0,0.0,0.0
2025-01-31 22:00:00,1,0,22,5,1,true,false,0,0,0,0.0,0.0


In [88]:
from pathlib import Path

In [92]:
df = result.sort(["PULocationID", "pickup_hour"])

zone1 = df.filter(pl.col("PULocationID") == 1)
print(zone1.head(30))
print("trip_count values:", zone1["trip_count"].head(30).to_list())

shape: (30, 3)
┌─────────────────────┬──────────────┬────────────┐
│ pickup_hour         ┆ PULocationID ┆ trip_count │
│ ---                 ┆ ---          ┆ ---        │
│ datetime[μs]        ┆ i64          ┆ u32        │
╞═════════════════════╪══════════════╪════════════╡
│ 2025-01-01 00:00:00 ┆ 1            ┆ 0          │
│ 2025-01-01 01:00:00 ┆ 1            ┆ 0          │
│ 2025-01-01 02:00:00 ┆ 1            ┆ 0          │
│ 2025-01-01 03:00:00 ┆ 1            ┆ 0          │
│ 2025-01-01 04:00:00 ┆ 1            ┆ 0          │
│ …                   ┆ …            ┆ …          │
│ 2025-01-02 01:00:00 ┆ 1            ┆ 0          │
│ 2025-01-02 02:00:00 ┆ 1            ┆ 0          │
│ 2025-01-02 03:00:00 ┆ 1            ┆ 0          │
│ 2025-01-02 04:00:00 ┆ 1            ┆ 0          │
│ 2025-01-02 05:00:00 ┆ 1            ┆ 0          │
└─────────────────────┴──────────────┴────────────┘
trip_count values: [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 

In [93]:
zone1 = df.filter(pl.col("PULocationID") == 1)
print("total zone 1 rows:", len(zone1))
print("zero trip_count rows:", len(zone1.filter(pl.col("trip_count") == 0)))
print("non-zero trip_count rows:", len(zone1.filter(pl.col("trip_count") > 0)))
print(zone1.head(10))

total zone 1 rows: 745
zero trip_count rows: 682
non-zero trip_count rows: 63
shape: (10, 3)
┌─────────────────────┬──────────────┬────────────┐
│ pickup_hour         ┆ PULocationID ┆ trip_count │
│ ---                 ┆ ---          ┆ ---        │
│ datetime[μs]        ┆ i64          ┆ u32        │
╞═════════════════════╪══════════════╪════════════╡
│ 2025-01-01 00:00:00 ┆ 1            ┆ 0          │
│ 2025-01-01 01:00:00 ┆ 1            ┆ 0          │
│ 2025-01-01 02:00:00 ┆ 1            ┆ 0          │
│ 2025-01-01 03:00:00 ┆ 1            ┆ 0          │
│ 2025-01-01 04:00:00 ┆ 1            ┆ 0          │
│ 2025-01-01 05:00:00 ┆ 1            ┆ 0          │
│ 2025-01-01 06:00:00 ┆ 1            ┆ 1          │
│ 2025-01-01 07:00:00 ┆ 1            ┆ 0          │
│ 2025-01-01 08:00:00 ┆ 1            ┆ 0          │
│ 2025-01-01 09:00:00 ┆ 1            ┆ 0          │
└─────────────────────┴──────────────┴────────────┘


In [94]:
print("zone 1 total rows:", len(zone1))

# manually compute what rolling_mean_24h should be for first 30 rows
trip_counts = zone1["trip_count"].head(30).to_list()
print("raw trip_counts:", trip_counts)

# shift by 1 then rolling mean of 24
shifted = [None] + trip_counts[:-1]
print("shifted:", shifted[:30])

# rolling mean of 24 on shifted
for i in range(30):
    window = shifted[max(0,i-23):i+1]
    valid = [x for x in window if x is not None]
    mean = sum(valid)/len(valid) if valid else None
    print(f"row {i}: window_size={len(valid)}, mean={mean}")

zone 1 total rows: 745
raw trip_counts: [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
shifted: [None, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
row 0: window_size=0, mean=None
row 1: window_size=1, mean=0.0
row 2: window_size=2, mean=0.0
row 3: window_size=3, mean=0.0
row 4: window_size=4, mean=0.0
row 5: window_size=5, mean=0.0
row 6: window_size=6, mean=0.0
row 7: window_size=7, mean=0.14285714285714285
row 8: window_size=8, mean=0.125
row 9: window_size=9, mean=0.1111111111111111
row 10: window_size=10, mean=0.1
row 11: window_size=11, mean=0.09090909090909091
row 12: window_size=12, mean=0.08333333333333333
row 13: window_size=13, mean=0.07692307692307693
row 14: window_size=14, mean=0.14285714285714285
row 15: window_size=15, mean=0.2
row 16: window_size=16, mean=0.1875
row 17: window_size=17, mean=0.23529411764705882
row 18: window_size=18, mean=0.2222222222222222
row 19: window_size=19, mea